In [2]:
import langchain
import langchain.embeddings
import pinecone
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Pinecone
from langchain_openai import OpenAI

/nmhs2/hari/work/UAV/Projects/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
os.environ["http_proxy"] = "http://14.139.134.20:3128"
os.environ["https_proxy"] = "http://14.139.134.20:3128"
os.environ["HTTP_PROXY"] = "http://14.139.134.20:3128"
os.environ["HTTPS_PROXY"] = "http://14.139.134.20:3128"



In [4]:
from dotenv import load_dotenv
import os

load_dotenv()



True

In [5]:
from langchain_community.document_loaders import PyPDFLoader

def read_one_doc(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    return documents

docs = read_one_doc(
    "/nmhs2/hari/work/UAV/Projects/rag_pipeline/pdfs/AI_Engineering.pdf"
)

print(len(docs))          # number of pages
print(docs[0].metadata)   # source + page


535
{'producer': 'Antenna House PDF Output Library 2.6.0 (Linux64)', 'creator': 'AH CSS Formatter V6.0 MR2 for Linux64 : 6.0.2.5372 (2012/05/16 18:26JST)', 'creationdate': '2024-12-04T13:39:11+00:00', 'author': 'Chip Huyen;', 'moddate': '2024-12-04T09:21:26-05:00', 'title': 'AI Engineering', 'trapped': '/False', 'ebx_publisher': "O'Reilly Media", 'source': '/nmhs2/hari/work/UAV/Projects/rag_pipeline/pdfs/AI_Engineering.pdf', 'total_pages': 535, 'page': 0, 'page_label': 'Cover'}


In [6]:
def chunk_data(docs,chunk_size=1000,chunk_overlap=100):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
    doc=text_splitter.split_documents(docs)
    return docs

In [7]:
documents=chunk_data(docs=docs)
len(documents)

535

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 522.16it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [11]:
vectors=embeddings.embed_query("How are you?")
len(vectors)

384